In [1]:
import pandas as pd
import win32com.client as win32
from pathlib import Path
from datetime import datetime
from html import escape

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:

import pandas as pd
from pathlib import Path

# =====================================
# 1. DEFERRAL FOLDER LOCATION
# =====================================

folder_path = Path(
    r"C:\Users\hp\OneDrive\Desktop\DEFERRALS"
)

# =====================================
# 2. FIND ALL EXCEL FILES
# =====================================

excel_files = list(folder_path.glob("*.xlsx"))

# Also include older .xls files
excel_files += list(folder_path.glob("*.xls"))

# Exclude temporary Excel files
excel_files = [
    file for file in excel_files
    if not file.name.startswith("~$")
]

# =====================================
# 3. IDENTIFY THE MOST RECENT FILE
# =====================================

if not excel_files:
    raise FileNotFoundError(
        "No Excel files found in the DEFERRALS folder."
    )

latest_file = max(
    excel_files,
    key=lambda file: file.stat().st_mtime
)

print("Most recent deferral file:")
print(latest_file.name)

print("Last modified:")
print(pd.to_datetime(
    latest_file.stat().st_mtime,
    unit="s"
))

# =====================================
# 4. LOAD THE LATEST EXCEL FILE
# =====================================

df = pd.read_excel(latest_file)

# Clean column headings
df.columns = df.columns.str.strip()

# Display data
display(df.head())

print("\nDATA LOADED SUCCESSFULLY")
print("Total records:", len(df))


Most recent deferral file:
Deferral_LIST AS AT 11.18.2026.xlsx
Last modified:
2026-11-18 18:43:51.839123011


,Deferral date,REQUESTER/RM,Borrower,Customer Number,Business Segment,Item Deferred,Item deffered group,Deferral Categorization,Risk classification,Proposed Legal Risk,...,Classification/Stage,EXPOSURE KES'000,FEEDBACK,CURRENT RM,BRANCH,REGION,Classification2,SUB SEGMENT,RM Email,BM Email
0,2025-01-14,Kenneth Njeru,Mary Wanjiku Kamau,779961,Retail Banking - Platinum Banking,Share certificate,Share certificate,PRIMARY CATEGORY B,Medium Risk,Medium Risk,...,NORMAL,9058,NaN,KENNETH NJERU,"SWISSPORT BRANCH, NAIROBI",NAIROBI,A5,PLATINUM,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com
1,2025-02-07,Kenneth Njeru,Peter Mwangi Maina,317578,Retail Banking - Gold Banking,Original logbook,Security documents,PRIMARY CATEGORY A,High Risk,High Risk,...,NORMAL,11433,Customer contacted; awaiting document.,KENNETH NJERU,THIKA ROAD MALL BRANCH,NAIROBI,A5,GOLD,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com
2,2025-03-29,Kenneth Njeru,Grace Nyambura Kariuki,452954,Retail Banking - Platinum Banking,Share certificate,Share certificate,PRIMARY CATEGORY B,Medium Risk,Medium Risk,...,NORMAL,6544,NaN,KENNETH NJERU,EMBAKASI BRANCH,EASTERN,A5,GOLD,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com
3,2025-04-11,Kenneth Njeru,David Kamau Njoroge,665770,Retail Banking - Platinum Banking,Title deed,Security documents,PRIMARY CATEGORY A,High Risk,High Risk,...,NORMAL,2869,Legal team follow-up required.,KENNETH NJERU,VILLAGE MARKET BRANCH,NAIROBI,A3,GOLD,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com
4,2025-05-02,Kenneth Njeru,Ann Muthoni Githua,812345,Retail Banking - Personal Banking,Discharge of charge,Security documents,PRIMARY CATEGORY B,Medium Risk,Medium Risk,...,NORMAL,4720,NaN,KENNETH NJERU,RUAKA BRANCH,NAIROBI,A4,PERSONAL,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com



DATA LOADED SUCCESSFULLY
Total records: 10


In [3]:

# Check available statuses
print(df["Status"].value_counts(dropna=False))

# Filter pending deferrals
pending_df = df[
    df["Status"].astype(str).str.strip().str.lower() == "pending"
].copy()

print("Total pending deferrals:", len(pending_df))

display(
    pending_df[
        ["Borrower", "Customer Number", "Item Deferred",
         "RM Email", "BM Email", "Status"]
    ]
)


Status
Pending    10
Name: count, dtype: int64
Total pending deferrals: 10


,Borrower,Customer Number,Item Deferred,RM Email,BM Email,Status
0,Mary Wanjiku Kamau,779961,Share certificate,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
1,Peter Mwangi Maina,317578,Original logbook,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
2,Grace Nyambura Kariuki,452954,Share certificate,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
3,David Kamau Njoroge,665770,Title deed,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
4,Ann Muthoni Githua,812345,Discharge of charge,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
5,John Karanja Mutua,823456,Insurance policy,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
6,Lucy Wairimu Njoki,834567,Valuation report,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
7,Samuel Kiptoo Rono,845678,Guarantee form,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
8,Esther Wambui Njeri,856789,Share certificate,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending
9,Brian Otieno Ouma,867890,Insurance policy,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Pending


In [4]:

# Connect to Outlook
outlook = win32.Dispatch("Outlook.Application")

# Access Outlook session
namespace = outlook.GetNamespace("MAPI")

print("Successfully connected to Outlook!")


Successfully connected to Outlook!


In [5]:

# ==========================================
# EMAIL AUTOMATION CONFIGURATION
# ==========================================

# IMPORTANT:
# False = Save emails as drafts
# True  = Send emails automatically

SEND_EMAILS = True

# Email tracking
email_log = []

# ==========================================
# EMAIL GENERATION FUNCTION
# ==========================================

def create_deferral_email(row):

    rm_email = str(row["RM Email"]).strip()
    bm_email = str(row["BM Email"]).strip()

    # Validate email addresses
    if (
        pd.isna(row["RM Email"])
        or pd.isna(row["BM Email"])
        or not rm_email
        or not bm_email
    ):
        raise ValueError("Missing RM or BM email address")

    borrower = escape(str(row["Borrower"]))
    customer = escape(str(row["Customer Number"]))
    item = escape(str(row["Item Deferred"]))
    branch = escape(str(row["BRANCH"]))
    requester = escape(str(row["REQUESTER/RM"]))

    due_date = pd.to_datetime(row["Due Date"]).strftime("%d %B %Y")

    ageing = row["Ageing (Days)"]

    exposure = pd.to_numeric(
        row["EXPOSURE KES'000"], errors="coerce"
    )

    exposure = f"{exposure:,.2f}" if pd.notna(exposure) else "N/A"

    # Create Outlook email
    mail = outlook.CreateItem(0)

    mail.To = rm_email
    mail.CC = bm_email

    mail.Subject = (
        f"Outstanding Deferral Follow-up | "
        f"{borrower} | {customer}"
    )

    # Email content
    mail.HTMLBody = f"""
    <html>
    <body style="font-family:Arial; font-size:11pt;">

    <p>Hi,</p>

    <p>
    Kindly assist in following up on the outstanding
    deferral item below and ensure that the required
    documentation is submitted to the bank.
    </p>

    <p><b>DEFERRAL DETAILS</b></p>

    <table border="1" cellpadding="7"
    cellspacing="0" style="border-collapse:collapse;">

    <tr><td><b>Borrower</b></td><td>{borrower}</td></tr>
    <tr><td><b>Customer Number</b></td><td>{customer}</td></tr>
    <tr><td><b>Relationship Manager</b></td><td>{requester}</td></tr>
    <tr><td><b>Branch</b></td><td>{branch}</td></tr>
    <tr><td><b>Deferred Item</b></td><td>{item}</td></tr>
    <tr><td><b>Due Date</b></td><td>{due_date}</td></tr>
    <tr><td><b>Ageing</b></td><td>{ageing} Days</td></tr>
    <tr><td><b>Exposure (KES'000)</b></td><td>{exposure}</td></tr>

    </table>

    <p>
    The above deferral remains outstanding.
    Kindly follow up with the customer and the relevant
    parties to ensure that the required documentation
    is submitted.
    </p>

    <p>
    Please provide feedback on the progress made and
    the expected date of submission.
    </p>

    <p>
    Your support in ensuring timely closure of this
    outstanding item is highly appreciated.
    </p>

    <p>
    Regards,<br>
    Credit Monitoring Team
    </p>

    </body>
    </html>
    """

    # Save draft or send email
    if SEND_EMAILS:
        mail.Send()
        result = "Sent"
    else:
        mail.Save()
        result = "Draft Saved"

    return result


print("Email generation function is ready!")


Email generation function is ready!


In [6]:

for index, row in pending_df.iterrows():

    try:
        result = create_deferral_email(row)

        email_log.append({
            "Customer Number": row["Customer Number"],
            "Borrower": row["Borrower"],
            "RM Email": row["RM Email"],
            "BM Email": row["BM Email"],
            "Result": result,
            "Date Processed": datetime.now()
        })

        print(
            f"SUCCESS: {row['Borrower']} - {result}"
        )

    except Exception as e:

        email_log.append({
            "Customer Number": row["Customer Number"],
            "Borrower": row["Borrower"],
            "RM Email": row["RM Email"],
            "BM Email": row["BM Email"],
            "Result": f"Failed: {e}",
            "Date Processed": datetime.now()
        })

        print(f"ERROR: {row['Borrower']} - {e}")


print("\nAUTOMATION COMPLETED")
print("Total processed:", len(email_log))


SUCCESS: Mary Wanjiku Kamau - Sent
SUCCESS: Peter Mwangi Maina - Sent
SUCCESS: Grace Nyambura Kariuki - Sent
SUCCESS: David Kamau Njoroge - Sent
SUCCESS: Ann Muthoni Githua - Sent
SUCCESS: John Karanja Mutua - Sent
SUCCESS: Lucy Wairimu Njoki - Sent
SUCCESS: Samuel Kiptoo Rono - Sent
SUCCESS: Esther Wambui Njeri - Sent
SUCCESS: Brian Otieno Ouma - Sent

AUTOMATION COMPLETED
Total processed: 10


In [7]:

# Convert email log into DataFrame
email_report = pd.DataFrame(email_log)

# Save tracking report
log_path = Path.home() / "Downloads" / "Deferral_Email_Log.xlsx"

email_report.to_excel(log_path, index=False)

# Display report
display(email_report)

print("Email log saved to:", log_path)


,Customer Number,Borrower,RM Email,BM Email,Result,Date Processed
0,779961,Mary Wanjiku Kamau,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:32.233145
1,317578,Peter Mwangi Maina,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:32.596030
2,452954,Grace Nyambura Kariuki,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:33.427004
3,665770,David Kamau Njoroge,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:34.651556
4,812345,Ann Muthoni Githua,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:35.426643
5,823456,John Karanja Mutua,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:36.149937
6,834567,Lucy Wairimu Njoki,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:36.577510
7,845678,Samuel Kiptoo Rono,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:37.220193
8,856789,Esther Wambui Njeri,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:37.562845
9,867890,Brian Otieno Ouma,kenneth.njeru@ncbagroup.com,kenneth.njeru@ncbagroup.com,Sent,2026-11-19 05:30:38.077003


Email log saved to: C:\Users\hp\Downloads\Deferral_Email_Log.xlsx
